In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
import os
import shutil

c:\Users\alons\miniconda3\envs\rag-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Rutas de los PDFs
pdf_files = [
    "data/LoRA-paper.pdf",
    "data/QLoRA-paper.pdf",
    "data/RAG-paper.pdf",
    "data/DARE-TIES-paper.pdf",
]

# Cargar y dividir en chunks
documentos = []
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    pages = loader.load()
    chunks = text_splitter.split_documents(pages)
    documentos.extend(chunks)
    print(f"hecho! {pdf} → {len(chunks)} chunks")

print(f"\nTotal chunks: {len(documentos)}")

hecho! data/LoRA-paper.pdf → 104 chunks
hecho! data/QLoRA-paper.pdf → 109 chunks
hecho! data/RAG-paper.pdf → 88 chunks
hecho! data/DARE-TIES-paper.pdf → 84 chunks

Total chunks: 385


In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

if os.path.exists("chroma_db"):
    vectordb = Chroma(
        persist_directory="chroma_db",
        embedding_function=embeddings
    )
    print(f"ChromaDB cargado con {vectordb._collection.count()} vectores")
else:
    vectordb = Chroma.from_documents(
        documents=documentos,
        embedding=embeddings,
        persist_directory="chroma_db"
    )
    print(f"ChromaDB creado con {vectordb._collection.count()} vectores")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4749.38it/s]


ChromaDB cargado con 713 vectores


Conexión con Ollama y construcción del pipeline

In [12]:
# Conectar con Mistral via Ollama
llm = Ollama(model="mistral")

# Configurar el retriever
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Ejemplo de prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

Question: {question}

Answer in a clear and concise way:
""")

# Pipeline RAG
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG pipeline ready")

RAG pipeline ready


In [15]:
def preguntar(pregunta):
    print(f"Question: {pregunta}")
    print("-" * 50)
    respuesta = rag_chain.invoke(pregunta)
    print(f"Answer: {respuesta}")
    print("-" * 50)

pregunta = input("Make your question: ")
preguntar(pregunta)

Question: What problem does Retrieval Augmented Generation solve and how does it work?
--------------------------------------------------
Answer:  The Retrieval Augmented Generation (RAG) solves the problem of generating language that is more factual and specific, particularly in open-domain question answering. It works by combining pre-trained parametric memory (a seq2seq model) with non-parametric memory (a dense vector index of Wikipedia), which is accessed with a pre-trained neural retriever. This combination allows RAG models to generate responses that are more informed and accurate, as they can draw upon a vast knowledge base during the generation process.
--------------------------------------------------


In [16]:
def ver_chunks_recuperados(pregunta):
    chunks = retriever.invoke(pregunta)
    print(f"Chunks recuperados para: '{pregunta}'")
    print("-" * 50)
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1}:")
        print(chunk.page_content)
        print("-" * 50)

ver_chunks_recuperados(pregunta)

Chunks recuperados para: 'What problem does Retrieval Augmented Generation solve and how does it work?'
--------------------------------------------------
Chunk 1:
6 Discussion
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We conducted an thorough investigation of the learned retrieval component, validating
its effectiveness, and we illustrated how the retrieval index can be hot-swapped to update the model
--------------------------------------------------
Chunk 2:
retrieval, more recently with pre-trained, neural language models [ 44, 26] similar to ours. Some
work optimizes the retrieval module to aid in a speciﬁc, downstream task such as question answering,
using search [46], reinforcement learning [6, 63, 62], or a latent variable appr